In [28]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [33]:
DATA_DIR = Path("/content/drive/MyDrive/share_data")
TABLE_DIR = DATA_DIR / "da-dev-tables"

# Project 3 Part B

## Task 1 Dataset Inspection and Sanity Checks

### Task 1.1 Load and Inspect JSONL Files

In [38]:
import json
from pathlib import Path

# ====== SETUP: point this to your dataset folder ======
# Expected layout (per spec):
#   da-dev-questions.jsonl
#   da-dev-labels.jsonl
#   da-dev-tables/   (used later)
DATA_DIR = Path("./share_data")  # change if your files are elsewhere

q_path = DATA_DIR / "da-dev-questions.jsonl"
l_path = DATA_DIR / "da-dev-labels.jsonl"

def load_jsonl(path: Path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

questions = load_jsonl(q_path)
labels = load_jsonl(l_path)

# ====== Q14 outputs ======
print("Number of questions:", len(questions))
print("Number of labels:", len(labels))

print("\n--- Question record keys ---")
print(sorted(list(questions[0].keys())))
print("\n--- One example question record ---")
print(json.dumps(questions[0], indent=2, ensure_ascii=False)[:2000])  # truncate for readability

print("\n--- Label record keys ---")
print(sorted(list(labels[0].keys())))
print("\n--- One example label record ---")
print(json.dumps(labels[0], indent=2, ensure_ascii=False)[:2000])     # truncate for readability

Number of questions: 257
Number of labels: 257

--- Question record keys ---
['concepts', 'constraints', 'file_name', 'format', 'id', 'level', 'question']

--- One example question record ---
{
  "id": 0,
  "question": "Calculate the mean fare paid by the passengers.",
  "concepts": [
    "Summary Statistics"
  ],
  "constraints": "Calculate the mean fare using Python's built-in statistics module or appropriate statistical method in pandas. Rounding off the answer to two decimal places.",
  "format": "@mean_fare[mean_fare_value] where \"mean_fare_value\" is a floating-point number rounded to two decimal places.",
  "file_name": "test_ave.csv",
  "level": "easy"
}

--- Label record keys ---
['common_answers', 'id']

--- One example label record ---
{
  "id": 0,
  "common_answers": [
    [
      "mean_fare",
      "34.65"
    ]
  ]
}


#### Question 14

The dataset contains 257 questions and 257 labels, indicating a one-to-one correspondence between questions and their labels.

Each question record contains the following keys: `concepts`, `constraints`, `file_name`, `format`, `id`, `level`, and `question`. An example question asks to calculate the mean fare paid by passengers, specifies summary statistics as the concept, includes rounding constraints, defines the expected output format, references a CSV file, and is labeled as easy.

Each label record contains the keys `id` and `common_answers`. The `id` corresponds to the question ID, and `common_answers` provides acceptable answers for evaluation.

### Task 1.2 Inspect the Table References

#### Question 15

In [42]:
import random
import pandas as pd

# Select 3 random question IDs
random_ids = random.sample(range(len(questions)), 3)

results = []

for qid in random_ids:
    q = questions[qid]
    csv_file = q["file_name"]
    csv_path = TABLE_DIR / csv_file
    
    print("="*70)
    print(f"Question ID: {qid}")
    print(f"Referenced CSV: {csv_file}")
    
    df = pd.read_csv(csv_path)
    
    print("\nShape:", df.shape)
    print("\nDtypes:\n", df.dtypes)
    print("\nHead (3 rows):\n", df.head(3))
    
    print("\nQuestion:")
    print(q["question"])
    
    results.append((qid, csv_file, df.shape, list(df.columns)))

Question ID: 88
Referenced CSV: 3901.csv

Shape: (1153, 19)

Dtypes:
 TRUE_TIME     object
TIME         float64
USFLUX       float64
MEANGAM      float64
MEANGBT      float64
MEANGBZ      float64
MEANGBH      float64
MEANJZD      float64
TOTUSJZ      float64
MEANJZH      float64
TOTUSJH      float64
ABSNJZH      float64
SAVNCPP      float64
MEANPOT      float64
TOTPOT       float64
MEANSHR       object
SHRGT45      float64
R_VALUE      float64
AREA_ACR     float64
dtype: object

Head (3 rows):
                  TRUE_TIME  TIME        USFLUX  MEANGAM  MEANGBT  MEANGBZ  \
0  2014.03.23_20:24:00_TAI  11.6  3.246502e+21   21.786   93.013   92.809   
1  2014.03.23_20:36:00_TAI  11.8  3.908340e+21   21.740   89.953   89.779   
2  2014.03.23_20:48:00_TAI  12.0  4.041844e+21   21.797   89.552   89.566   

   MEANGBH   MEANJZD       TOTUSJZ   MEANJZH  TOTUSJH  ABSNJZH       SAVNCPP  \
0   31.210  0.087461  3.141588e+12  0.002863  143.341   14.092  2.248874e+11   
1   31.535  0.151386  3.745627e

### Task 1.3 Understand the Required Answer Format

#### Question 16

Two examples where the required format contains multiple answer slots are **Question ID 275** and **Question ID 144**.  
For **ID 275**, the required format has **four** slots: `@duplicate_count[...]`, `@usflux_mean[...]`, `@new_feature_mean[...]`, and `@model_accuracy[...]`. For **ID 144**, the format has **six** slots: `@mean_dem[...]`, `@mean_gop[...]`, `@std_dev_dem[...]`, `@std_dev_gop[...]`, `@dist_dem[...]`, and `@dist_gop[...]`.

**How the dataset represents multi-part answers in the labels:** multi-part answers are stored in each label record’s `common_answers` as **multiple acceptable answer variants**, where each variant consists of **(slot_name, value)** pairs (e.g., for ID 275 there are 2 variants; for ID 144 there are 4 variants). This structure allows multiple valid completions while still keeping each required field explicitly named.

**How to evaluate such answers automatically:** parse the model output to extract all `@slot[value]` pairs, then (1) verify that the set of slots matches the required slots from the question’s `format`, (2) normalize values (strip whitespace; parse numbers; enforce rounding/tolerance rules; canonicalize strings like `"Normal"`/`"Not Normal"`), and (3) compare the resulting slot→value mapping against each acceptable variant in `common_answers`. Mark the answer correct if it matches **any** variant, and optionally allow order-insensitivity since slots are explicitly named.

In [44]:
import re
import random
import json

slot_pat = re.compile(r'@([A-Za-z0-9_]+)\[')

# Build id->record maps (safe even if ids aren't 0..N-1 or not aligned)
q_by_id = {q["id"]: q for q in questions}
l_by_id = {l["id"]: l for l in labels}

multi = []
for q in questions:
    slots = slot_pat.findall(q["format"])
    if len(slots) >= 2:
        multi.append((q["id"], slots))

print("Num questions with >=2 answer slots:", len(multi))

# pick 2 examples
example_ids = [qid for qid, _ in random.sample(multi, 2)]

for qid in example_ids:
    q = q_by_id[qid]
    lab = l_by_id[qid]
    slots = slot_pat.findall(q["format"])

    print("=" * 80)
    print("Question ID:", qid)
    print("Question:", q["question"])
    print("Format:", q["format"])
    print("Slots (in order):", slots)

    ca = lab["common_answers"]
    print("\nLabel common_answers: num variants =", len(ca))
    print("First 2 variants (truncated):")
    print(json.dumps(ca[:2], indent=2, ensure_ascii=False)[:1500])

Num questions with >=2 answer slots: 145
Question ID: 275
Question: Perform a comprehensive analysis of the dataset by:
1. Removing any duplicate entries.
2. Filling in missing values in the USFLUX column with the mean value of the column.
3. Creating a new feature named "MEANGAM_MEANGBZ_DIFF" by subtracting the MEANGBZ column from the MEANGAM column.
4. Applying machine learning techniques to predict the values in the TOTUSJH column using the MEANJZH, TOTUSJZ, and MEANGBT columns. You will need to use a Random Forest Regressor with 100 trees for this task.
Format: 1. @duplicate_count[duplicate_total] where "duplicate_total" should be an integer indicating the number of duplicate rows removed.
2. @usflux_mean[mean_value] where "mean_value" should be a number rounded to 2 decimal places.
3. @new_feature_mean[new_feature_mean] where "new_feature_mean" is the mean of the new feature "MEANGAM_MEANGBZ_DIFF", rounded to 2 decimal places.
4. @model_accuracy[model_accuracy] where "model_accura

### Task 1.4 Checking the subset

#### Question 17

The selected subset of question IDs is:
[0, 5, 9, 10, 14, 18, 24, 25, 26, 55].

All selected tasks were printed and verified. Each ID exists in the dataset and references a valid CSV file. The tasks primarily involve summary statistics (mean, standard deviation), correlation analysis, distribution testing (normality checks), and basic feature engineering.

Most selected questions are labeled as *easy*, with only two marked as *medium*. The required answer formats are clearly structured with one or a small number of answer slots, and the computations involve straightforward statistical operations rather than complex multi-step modeling or advanced machine learning.

Overall, this subset consists of well-defined, numerically grounded tasks that are feasible for the model to solve, justifying their selection as solvable sub-tasks.

In [45]:
SELECTED_IDS = [0, 5, 9, 10, 14, 18, 24, 25, 26, 55]

# Build id -> question mapping (safe)
q_by_id = {q["id"]: q for q in questions}

print("Checking selected tasks:\n")

for qid in SELECTED_IDS:
    print("=" * 80)
    print(f"Question ID: {qid}")
    
    if qid in q_by_id:
        q = q_by_id[qid]
        print("Level:", q["level"])
        print("Concepts:", q["concepts"])
        print("File:", q["file_name"])
        print("Format:", q["format"])
        print("\nQuestion:")
        print(q["question"])
    else:
        print("ID not found in dataset.")

Checking selected tasks:

Question ID: 0
Level: easy
Concepts: ['Summary Statistics']
File: test_ave.csv
Format: @mean_fare[mean_fare_value] where "mean_fare_value" is a floating-point number rounded to two decimal places.

Question:
Calculate the mean fare paid by the passengers.
Question ID: 5
Level: medium
Concepts: ['Feature Engineering', 'Correlation Analysis']
File: test_ave.csv
Format: @correlation_coefficient[r_value]
where "r_value" is the Pearson correlation coefficient between 'FamilySize' and 'Fare', a number between -1 and 1, rounded to two decimal places.

Question:
Generate a new feature called "FamilySize" by summing the "SibSp" and "Parch" columns. Then, calculate the Pearson correlation coefficient (r) between the "FamilySize" and "Fare" columns.
Question ID: 9
Level: easy
Concepts: ['Summary Statistics']
File: GODREJIND.csv
Format: @mean_close_price[mean_value], where "mean_value" is a float number rounded to two decimal places. This value should be between the highe